# ThetaData research smoke test

Loads SPY, the SPY option chain of 2026-10-23, and the daily history of 1 call, from ThetaData end-of-day data (Free plan).
Start it with `lean research` and the ThetaData options that `LOCAL_LEAN_SETUP.md` gives.

In [ ]:
from AlgorithmImports import *

qb = QuantBook()
qb.set_start_date(2026, 10, 24)
spy = qb.add_equity("SPY", Resolution.DAILY).symbol

bars = qb.history(spy, datetime(2026, 10, 19), datetime(2026, 10, 24), Resolution.DAILY)
print(bars[["open", "high", "low", "close", "volume"]])

In [ ]:
chain = qb.option_chain(spy, flatten=True).data_frame
print(f"contracts: {len(chain)}, expirations: {chain['expiry'].nunique()}")
print(chain.head())

In [ ]:
underlying_price = bars["close"].iloc[-1]
calls = chain[(chain["right"] == OptionRight.CALL) & (chain["expiry"] == datetime(2026, 10, 30))]
contract = calls.iloc[(calls["strike"] - underlying_price).abs().argsort()].index[0]
print(f"contract: {contract}")

contract_history = qb.history(contract, datetime(2026, 10, 19), datetime(2026, 10, 24), Resolution.DAILY)
print(contract_history[[c for c in ["bidclose", "askclose", "close", "volume"] if c in contract_history.columns]])